In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import json

warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE ARCHITECTURE (WITHOUT FINAL FCN)
# ==========================================
# Paste your standard Graph, SpatialGraphConv, ChannelAttention, and STGCN_Block here
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
    def forward(self, x): 
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

# 🔴 MODIFIED: Extracts Embeddings instead of Classifying
class EmbeddingSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        
        # We keep this for loading weights, but we WON'T use it in the forward pass
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        # 🔴 Return the flattened 256-dimensional vector, ignore fcn
        return x.view(x.size(0), -1)

# ==========================================
# 2. DATABASE BUILDER ENGINE
# ==========================================
def build_prototype_database():
    TRAIN_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1\train"
    MODEL_WEIGHTS = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\7.9_layer_Attention_Models\9_layer_attention_relu_9frame.pth" 
    
    dml = torch_directml.device()
    print(f"\n🚀 DATABASE BUILDER STARTED. Using GPU: {torch_directml.device_name(0)}")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)

    graph = Graph()
    model = EmbeddingSTGCN(num_classes, graph.A).to(dml)
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=dml, weights_only=False))
    model.eval()

    prototypes = {}

    print("⏳ Extracting Embeddings for all Training Data...")
    with torch.no_grad():
        for class_name in all_classes:
            class_path = os.path.join(TRAIN_DIR, class_name)
            class_embeddings = []
            
            for f_name in os.listdir(class_path):
                if f_name.endswith('.npy'):
                    f_path = os.path.join(class_path, f_name)
                    raw_data = np.load(f_path).reshape(90, 68, 3) 
                    raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                    raw_data = raw_data.transpose(2, 0, 1)
                    
                    tensor_seq = torch.tensor(raw_data, dtype=torch.float32).unsqueeze(0).to(dml)
                    
                    # Get the 256-d vector
                    embedding = model(tensor_seq).cpu().numpy().squeeze()
                    class_embeddings.append(embedding)
            
            # Average all vectors for this class
            avg_embedding = np.mean(class_embeddings, axis=0)
            prototypes[class_name] = avg_embedding
            print(f"   ✅ Computed Matrix for: {class_name} (from {len(class_embeddings)} videos)")

    # Save the database
    matrix_data = []
    labels_data = []
    for cls_name, emb in prototypes.items():
        matrix_data.append(emb)
        labels_data.append(cls_name)

    np.save("class_prototypes.npy", np.array(matrix_data))
    with open("prototype_labels.json", "w") as f:
        json.dump(labels_data, f)
        
    print("\n🎉 SUCCESS! Matrix Database Saved:")
    print("- 'class_prototypes.npy'")
    print("- 'prototype_labels.json'")

if __name__ == "__main__":
    build_prototype_database()


🚀 DATABASE BUILDER STARTED. Using GPU: AMD Radeon RX 7900 GRE 
⏳ Extracting Embeddings for all Training Data...
   ✅ Computed Matrix for: 000_Lefthand (from 100 videos)
   ✅ Computed Matrix for: 000_Righthand (from 100 videos)
   ✅ Computed Matrix for: 00_Lefthand (from 100 videos)
   ✅ Computed Matrix for: 00_Righthand (from 100 videos)
   ✅ Computed Matrix for: 0_Lefthand (from 100 videos)
   ✅ Computed Matrix for: 0_Righthand (from 100 videos)
   ✅ Computed Matrix for: 1_Lefthand (from 100 videos)
   ✅ Computed Matrix for: 1_Righthand (from 100 videos)
   ✅ Computed Matrix for: 2_Lefthand (from 100 videos)
   ✅ Computed Matrix for: 2_Righthand (from 100 videos)
   ✅ Computed Matrix for: 3_Lefthand (from 100 videos)
   ✅ Computed Matrix for: 3_Righthand (from 100 videos)
   ✅ Computed Matrix for: 4_Lefthand (from 100 videos)
   ✅ Computed Matrix for: 4_Righthand (from 100 videos)
   ✅ Computed Matrix for: 5_Lefthand (from 100 videos)
   ✅ Computed Matrix for: 5_Righthand (from 100 v

Testing the database system

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import torch_directml
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, roc_auc_score
from sklearn.preprocessing import LabelBinarizer
import warnings

warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. GRAPH & EMBEDDING NETWORK ARCHITECTURE
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [(0,1), (1,2), (2,3), (3,7), (0,4), (4,5), (5,6), (6,8), (9,10),
                      (11,12), (11,23), (12,24), (23,24), (11,13), (13,15), (12,14), (14,16),
                      (15,17), (15,19), (15,21), (16,18), (16,20), (16,22)]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4), (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12), (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges: A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        print(f"📂 Loading data from: {os.path.basename(split_dir)}...")
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        self.samples.append((raw_data, cid))
        print(f"   ✅ Cached {len(self.samples)} samples in memory.")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        data, label = self.samples[idx]
        return torch.tensor(data, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 MUST INCLUDE THE ATTENTION MODULE
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True), 
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)), 
            nn.BatchNorm2d(out_c),
            nn.Dropout(dropout)
        )
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
    def forward(self, x): 
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

# 🔴 EMBEDDING EXTRACTOR (No final FCN)
class EmbeddingSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        
        # Kept for strict state_dict matching, not used in forward
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return x.view(x.size(0), -1) # Returns 256-dimensional structural feature representation

# ==========================================
# 2. METRIC ANALYSIS & REPORT EXPORTER
# ==========================================
def calculate_and_save_metrics(y_true, y_pred, y_scores, labels, output_dir, prefix):
    acc = np.mean(np.array(y_true) == np.array(y_pred)) * 100
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    
    # Calculate Multi-Class ROC AUC (One-Vs-Rest)
    try:
        lb = LabelBinarizer()
        lb.fit(labels)
        y_true_bin = lb.transform(y_true)
        if y_true_bin.shape[1] == y_scores.shape[1]:
            roc_auc = roc_auc_score(y_true_bin, y_scores, average='macro', multi_class='ovr')
        else:
            roc_auc = 0.0
    except:
        roc_auc = 0.0

    # Save Confusion Matrix CSV
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    cm_df.to_csv(os.path.join(output_dir, f"{prefix}_prototype_confusion_matrix.csv"))

    # Save Text Report
    report_txt = classification_report(y_true, y_pred, labels=labels, zero_division=0)
    report_path = os.path.join(output_dir, f"{prefix}_prototype_classification_report.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"=== {prefix.upper()} RUN PROTOTYPICAL MATRIX RESULTS ===\n")
        f.write(f"Accuracy: {acc:.2f}%\n")
        f.write(f"Macro Precision: {p*100:.2f}%\n")
        f.write(f"Macro Recall: {r*100:.2f}%\n")
        f.write(f"Macro F1-Score: {f1*100:.2f}%\n")
        f.write(f"Multi-Class OVR ROC AUC: {roc_auc:.4f}\n\n")
        f.write(report_txt)
        
    print(f"\n📊 Results Generated for {prefix.upper()}:")
    print(f"   ↳ Accuracy: {acc:.2f}% | F1-Score: {f1*100:.2f}% | ROC AUC: {roc_auc:.4f}")
    print(f"   ↳ Saved: {prefix}_prototype_confusion_matrix.csv & {prefix}_prototype_classification_report.txt")

# ==========================================
# 3. RUNTIME EVALUATION LOOP
# ==========================================
def run_prototypical_evaluation():
    # 🟢 EXACT CORRECT PATHS PROVIDED
    DB_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\database Saved"
    MODEL_WEIGHTS = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\7.9_layer_Attention_Models\9_layer_attention_relu_9frame.pth"
    
    # 🟢 ASSUMED DATASET PATHS (Double check these match your system)
    VAL_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1\val"
    TEST_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Global_Test_Set"

    dml = torch_directml.device()
    print(f"\n🚀 MATRIX PROTOTYPE INFERENCE ENGINE BOOTED. Device: {torch_directml.device_name(0)}")

    # Load matrix database information from DB_DIR
    prototypes_matrix = np.load(os.path.join(DB_DIR, "class_prototypes.npy")) # Shape (342, 256)
    with open(os.path.join(DB_DIR, "prototype_labels.json"), "r") as f:
        all_classes = json.load(f)
        
    class_to_id = {name: idx for idx, name in enumerate(all_classes)}
    id_to_class = {idx: name for idx, name in enumerate(all_classes)}
    num_classes = len(all_classes)
    
    prototypes_tensor = torch.tensor(prototypes_matrix, dtype=torch.float32).to(dml)

    # Initialize model setup (Loading the 9-layer Attention model)
    model = EmbeddingSTGCN(num_classes, Graph().A).to(dml)
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=dml, weights_only=False))
    model.eval()

    # Load Splits
    val_dataset = CleanBanglaDataset(VAL_DIR, class_to_id)
    test_dataset = CleanBanglaDataset(TEST_DIR, class_to_id)
    
    loaders = {
        'validation': DataLoader(val_dataset, batch_size=32, shuffle=False),
        'test': DataLoader(test_dataset, batch_size=32, shuffle=False)
    }

    results = {'validation': {'true': [], 'pred': [], 'scores': []},
               'test': {'true': [], 'pred': [], 'scores': []}}

    print("\n🧠 Beginning Prototypical Space Inference via Tensor Distances...")
    with torch.no_grad():
        for split_name, loader in loaders.items():
            for inputs, labels in loader:
                inputs = inputs.to(dml)
                
                # Extract deep structure 256-d embeddings vectors
                embeddings = model(inputs) 
                
                # Calculate distances
                distances = torch.cdist(embeddings, prototypes_tensor, p=2)
                
                # The closest class vector in space wins
                min_distances, pred_ids = torch.min(distances, dim=1)
                
                # Pseudo-probs for ROC
                pseudo_probs = F.softmax(-distances, dim=1).cpu().numpy()

                for i in range(labels.size(0)):
                    results[split_name]['true'].append(id_to_class[labels[i].item()])
                    results[split_name]['pred'].append(id_to_class[pred_ids[i].item()])
                    results[split_name]['scores'].append(pseudo_probs[i])

    # Generate isolated splits metrics (Saved to DB_DIR)
    calculate_and_save_metrics(results['validation']['true'], results['validation']['pred'], np.array(results['validation']['scores']), all_classes, DB_DIR, "validation")
    calculate_and_save_metrics(results['test']['true'], results['test']['pred'], np.array(results['test']['scores']), all_classes, DB_DIR, "test")

    # Combine data tracking objects
    combined_true = results['validation']['true'] + results['test']['true']
    combined_pred = results['validation']['pred'] + results['test']['pred']
    combined_scores = np.concatenate([results['validation']['scores'], results['test']['scores']], axis=0)

    # Generate combined evaluation outputs (Saved to DB_DIR)
    calculate_and_save_metrics(combined_true, combined_pred, combined_scores, all_classes, DB_DIR, "combined")
    print(f"\n✅ SYSTEM CHECK COMPLETE. Reports safely stored inside:\n {DB_DIR}\n")

if __name__ == "__main__":
    run_prototypical_evaluation()


🚀 MATRIX PROTOTYPE INFERENCE ENGINE BOOTED. Device: AMD Radeon RX 7900 GRE 
📂 Loading data from: val...
   ✅ Cached 8681 samples in memory.
📂 Loading data from: Global_Test_Set...
   ✅ Cached 6840 samples in memory.

🧠 Beginning Prototypical Space Inference via Tensor Distances...

📊 Results Generated for VALIDATION:
   ↳ Accuracy: 98.72% | F1-Score: 98.68% | ROC AUC: 0.9998
   ↳ Saved: validation_prototype_confusion_matrix.csv & validation_prototype_classification_report.txt

📊 Results Generated for TEST:
   ↳ Accuracy: 98.36% | F1-Score: 98.36% | ROC AUC: 1.0000
   ↳ Saved: test_prototype_confusion_matrix.csv & test_prototype_classification_report.txt

📊 Results Generated for COMBINED:
   ↳ Accuracy: 98.56% | F1-Score: 98.55% | ROC AUC: 0.9999
   ↳ Saved: combined_prototype_confusion_matrix.csv & combined_prototype_classification_report.txt

✅ SYSTEM CHECK COMPLETE. Reports safely stored inside:
 C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Exp

In [2]:
import os
import shutil
import random
from collections import defaultdict

# ==========================================
# 1. EXACT DIRECTORY MAPPING
# ==========================================
BASE_THESIS_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1"

# Your existing safe test set
OLD_GLOBAL_TEST = os.path.join(BASE_THESIS_DIR, "Global_Test_Set")

# The new directories we are going to create
NEW_FEW_SHOT_TRAIN = os.path.join(BASE_THESIS_DIR, "Experimental_30Shot_Train")
NEW_GLOBAL_TEST = os.path.join(BASE_THESIS_DIR, "Experimental_Global_Test")

# Explicit mapping of the classes to their exact source paths
CLASS_PATHS = {
    "Chikissha_Treatment_Lefthand": r"F:\2.Editing Datasets\NPY_Aligned_Tensors\Chikissha_Treatment\Chikissha_Treatment_Lefthand",
    "Chikissha_Treatment_Righthand": r"F:\2.Editing Datasets\NPY_Aligned_Tensors\Chikissha_Treatment\Chikissha_Treatment_Righthand",
    "Bhalo_Fine": r"F:\2.Editing Datasets\NPY_Aligned_Tensors\Bhalo_Fine",
    "Paribar_Family": r"F:\2.Editing Datasets\NPY_Aligned_Tensors\Paribar_Family"
}

# Hyperparameters
TEST_SAMPLES = 20
TRAIN_SAMPLES = 30  
ACTOR_DELIMITER = "_" # Assuming filenames are like "Actor1_Sign_01.npy". Change to "-" or " " if needed.

# ==========================================
# 2. ROUND-ROBIN ACTOR BALANCING ALGORITHM
# ==========================================
def get_balanced_split(file_list, target_count):
    """Pulls exactly `target_count` files, balancing evenly across all actors."""
    groups = defaultdict(list)
    
    # Group files by actor ID (the prefix before the first underscore)
    for f in file_list:
        actor_id = f.split(ACTOR_DELIMITER)[0]
        groups[actor_id].append(f)
        
    for k in groups:
        random.shuffle(groups[k]) # Shuffle each actor's files for randomness
        
    selected_files = []
    actors = list(groups.keys())
    random.shuffle(actors) 
    
    # Pull 1 file from each actor sequentially until we hit the target count
    actor_idx = 0
    while len(selected_files) < target_count:
        if all(len(groups[a]) == 0 for a in actors):
            break # Failsafe: ran out of files
            
        curr_actor = actors[actor_idx % len(actors)]
        if len(groups[curr_actor]) > 0:
            selected_files.append(groups[curr_actor].pop(0))
            
        actor_idx += 1
        
    # Return the selected files, and a flat list of all remaining unselected files
    remaining_files = [f for sublist in groups.values() for f in sublist]
    return selected_files, remaining_files

# ==========================================
# 3. EXECUTION ENGINE
# ==========================================
def setup_few_shot_experiment():
    print("🚀 BOOTING ACTOR-BALANCED FEW-SHOT SPLITTER...")
    
    os.makedirs(NEW_FEW_SHOT_TRAIN, exist_ok=True)
    
    # 1. Copy the old global test set
    if not os.path.exists(NEW_GLOBAL_TEST):
        print(f"\n📂 Copying existing Global Test Set to experimental folder...")
        shutil.copytree(OLD_GLOBAL_TEST, NEW_GLOBAL_TEST)
        print("   ✅ Copy complete.")
    else:
        print(f"\n⚠️ Experimental Test Set already exists. Continuing...")

    print("\n🧠 Processing Few-Shot Classes with Actor Balancing...")
    
    random.seed(42) # Seeded for reproducible splits
    
    for cls, source_cls_dir in CLASS_PATHS.items():
        if not os.path.exists(source_cls_dir):
            print(f"   ❌ ERROR: Path not found -> {source_cls_dir}")
            continue
            
        all_files = [f for f in os.listdir(source_cls_dir) if f.endswith('.npy')]
        total_files = len(all_files)
        
        if total_files < (TEST_SAMPLES + TRAIN_SAMPLES):
            print(f"   ❌ ERROR: {cls} only has {total_files} files. Need at least {TEST_SAMPLES + TRAIN_SAMPLES}. Skipping.")
            continue
            
        # Get balanced splits
        test_files, remaining_after_test = get_balanced_split(all_files, TEST_SAMPLES)
        train_files, _ = get_balanced_split(remaining_after_test, TRAIN_SAMPLES)
        
        # Determine actor distribution for printing
        test_actors = [f.split(ACTOR_DELIMITER)[0] for f in test_files]
        train_actors = [f.split(ACTOR_DELIMITER)[0] for f in train_files]
        unique_actors = len(set(test_actors + train_actors))
        
        # Create destination class folders
        test_dest_dir = os.path.join(NEW_GLOBAL_TEST, cls)
        train_dest_dir = os.path.join(NEW_FEW_SHOT_TRAIN, cls)
        
        os.makedirs(test_dest_dir, exist_ok=True)
        os.makedirs(train_dest_dir, exist_ok=True)
        
        # Execute copy
        for f in test_files:
            shutil.copy2(os.path.join(source_cls_dir, f), os.path.join(test_dest_dir, f))
            
        for f in train_files:
            shutil.copy2(os.path.join(source_cls_dir, f), os.path.join(train_dest_dir, f))
            
        print(f"   ✅ {cls}: Sent {TRAIN_SAMPLES} Train | {TEST_SAMPLES} Test. (Balanced across {unique_actors} Actors)")

    print(f"\n🎉 DATA SPLITTING COMPLETE!")
    print(f"   ↳ Prototype Builder Source: {NEW_FEW_SHOT_TRAIN}")
    print(f"   ↳ Global Evaluation Target: {NEW_GLOBAL_TEST}")

if __name__ == "__main__":
    setup_few_shot_experiment()

🚀 BOOTING ACTOR-BALANCED FEW-SHOT SPLITTER...

📂 Copying existing Global Test Set to experimental folder...
   ✅ Copy complete.

🧠 Processing Few-Shot Classes with Actor Balancing...
   ✅ Chikissha_Treatment_Lefthand: Sent 30 Train | 20 Test. (Balanced across 1 Actors)
   ✅ Chikissha_Treatment_Righthand: Sent 30 Train | 20 Test. (Balanced across 1 Actors)
   ✅ Bhalo_Fine: Sent 30 Train | 20 Test. (Balanced across 1 Actors)
   ✅ Paribar_Family: Sent 30 Train | 20 Test. (Balanced across 1 Actors)

🎉 DATA SPLITTING COMPLETE!
   ↳ Prototype Builder Source: C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Experimental_30Shot_Train
   ↳ Global Evaluation Target: C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Experimental_Global_Test


Making the experimental Database

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import torch_directml
import warnings

warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE ARCHITECTURE (WITHOUT FINAL FCN)
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
    def forward(self, x): 
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

class EmbeddingSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        
        # Kept strictly to load your pre-trained weights safely
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        # Return the flattened 256-dimensional structural feature representation
        return x.view(x.size(0), -1)

# ==========================================
# 2. EXPERIMENTAL DATABASE BUILDER ENGINE
# ==========================================
def build_incremental_database():
    # Directories
    BASE_TRAIN_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1\train"
    NEW_FEW_SHOT_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Experimental_30Shot_Train"
    OUTPUT_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\experimental Database Maker"
    MODEL_WEIGHTS = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\7.9_layer_Attention_Models\9_layer_attention_relu_9frame.pth" 
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 🟢 FIX: torch_directml.device() with no argument defaults to device index 0,
    # which on this machine is the integrated "AMD Radeon(TM) Graphics", not the
    # discrete RX 7900 GRE. Scan all devices and explicitly select the 7900 GRE.
    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        print(f"   -> GPU {i}: {gpu_name}")
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name:
            target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"✅ Selected GPU {target_idx}: {torch_directml.device_name(target_idx)}\n")

    # We determine the original number of classes to load the model properly
    base_classes = sorted([d for d in os.listdir(BASE_TRAIN_DIR) if os.path.isdir(os.path.join(BASE_TRAIN_DIR, d))])
    original_num_classes = len(base_classes)

    graph = Graph()
    model = EmbeddingSTGCN(original_num_classes, graph.A).to(dml)
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=dml, weights_only=False))
    model.eval()

    prototypes = {}

    def extract_prototypes_from_dir(directory_path, data_label):
        classes_in_dir = sorted([d for d in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, d))])
        print(f"\n⏳ Extracting Embeddings for {data_label} ({len(classes_in_dir)} classes)...")
        
        with torch.no_grad():
            for class_name in classes_in_dir:
                class_path = os.path.join(directory_path, class_name)
                class_embeddings = []
                
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        tensor_seq = torch.tensor(raw_data, dtype=torch.float32).unsqueeze(0).to(dml)
                        embedding = model(tensor_seq).cpu().numpy().squeeze()
                        class_embeddings.append(embedding)
                
                # Average all vectors to create the single prototype anchor
                if len(class_embeddings) > 0:
                    avg_embedding = np.mean(class_embeddings, axis=0)
                    prototypes[class_name] = avg_embedding
                    print(f"   ✅ Computed Anchor for: {class_name} (from {len(class_embeddings)} files)")

    # 1. Process the robust base classes
    extract_prototypes_from_dir(BASE_TRAIN_DIR, "Base Classes")
    
    # 2. Process the new 30-shot experimental classes
    extract_prototypes_from_dir(NEW_FEW_SHOT_DIR, "Novel 30-Shot Classes")

    # Save the unified database
    matrix_data = []
    labels_data = []
    
    # Sort alphabetically to maintain strict order
    for cls_name in sorted(prototypes.keys()):
        matrix_data.append(prototypes[cls_name])
        labels_data.append(cls_name)

    np.save(os.path.join(OUTPUT_DIR, "class_prototypes.npy"), np.array(matrix_data))
    with open(os.path.join(OUTPUT_DIR, "prototype_labels.json"), "w") as f:
        json.dump(labels_data, f)
        
    print("\n🎉 SUCCESS! Incremental Matrix Database Saved:")
    print(f"- Location: {OUTPUT_DIR}")
    print(f"- Total Classes in Database: {len(labels_data)}")

if __name__ == "__main__":
    build_incremental_database()


🔍 Scanning available DirectML GPUs...
   -> GPU 0: AMD Radeon(TM) Graphics 
   -> GPU 1: AMD Radeon RX 7900 GRE 
✅ Selected GPU 1: AMD Radeon RX 7900 GRE 


⏳ Extracting Embeddings for Base Classes (342 classes)...
   ✅ Computed Anchor for: 000_Lefthand (from 100 files)
   ✅ Computed Anchor for: 000_Righthand (from 100 files)
   ✅ Computed Anchor for: 00_Lefthand (from 100 files)
   ✅ Computed Anchor for: 00_Righthand (from 100 files)
   ✅ Computed Anchor for: 0_Lefthand (from 100 files)
   ✅ Computed Anchor for: 0_Righthand (from 100 files)
   ✅ Computed Anchor for: 1_Lefthand (from 100 files)
   ✅ Computed Anchor for: 1_Righthand (from 100 files)
   ✅ Computed Anchor for: 2_Lefthand (from 100 files)
   ✅ Computed Anchor for: 2_Righthand (from 100 files)
   ✅ Computed Anchor for: 3_Lefthand (from 100 files)
   ✅ Computed Anchor for: 3_Righthand (from 100 files)
   ✅ Computed Anchor for: 4_Lefthand (from 100 files)
   ✅ Computed Anchor for: 4_Righthand (from 100 files)
   ✅ Computed A

Testing The Experiment

In [6]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import torch_directml
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import LabelBinarizer
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. CORE ARCHITECTURE & DATASET
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [(0,1), (1,2), (2,3), (3,7), (0,4), (4,5), (5,6), (6,8), (9,10),
                      (11,12), (11,23), (12,24), (23,24), (11,13), (13,15), (12,14), (14,16),
                      (15,17), (15,19), (15,21), (16,18), (16,20), (16,22)]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4), (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12), (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges: A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), nn.Dropout(dropout)
        )
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
    def forward(self, x): 
        return F.relu(self.attention(self.tgcn(self.sgcn(x))) + self.res(x)) 

class EmbeddingSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return x.view(x.size(0), -1)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if os.path.isdir(class_path) and class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        self.samples.append((raw_data.transpose(2, 0, 1), cid))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return torch.tensor(self.samples[idx][0], dtype=torch.float32), torch.tensor(self.samples[idx][1], dtype=torch.long)

# ==========================================
# 2. LOGIC MAPPERS
# ==========================================
def map_to_relaxed(label):
    return label.replace("_Lefthand", "").replace("_Righthand", "").strip()

def map_to_base_sign(label):
    relaxed = map_to_relaxed(label)
    return relaxed.split('_')[0]

# ==========================================
# 3. EVALUATION MATRICES GENERATOR
# ==========================================
def generate_matrices():
    # PATHS
    DB_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\experimental Database Maker"
    EVAL_DIR = os.path.join(DB_DIR, "Evaluation Matrices")
    TEST_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Experimental_Global_Test"
    MODEL_WEIGHTS = r"C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\7.9_layer_Attention_Models\9_layer_attention_relu_9frame.pth"
    
    os.makedirs(EVAL_DIR, exist_ok=True)
    
    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        print(f"   -> GPU {i}: {gpu_name}")
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name:
            target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"✅ Selected GPU {target_idx}: {torch_directml.device_name(target_idx)}\n")
    
    print(f"🚀 STARTING ADVANCED EVALUATION ON NEW DATABASE...")

    # Load experimental DB
    prototypes_matrix = np.load(os.path.join(DB_DIR, "class_prototypes.npy"))
    with open(os.path.join(DB_DIR, "prototype_labels.json"), "r") as f:
        all_classes = json.load(f)
        
    class_to_id = {name: idx for idx, name in enumerate(all_classes)}
    id_to_class = {idx: name for idx, name in enumerate(all_classes)}
    prototypes_tensor = torch.tensor(prototypes_matrix, dtype=torch.float32).to(dml)

    # 🔴 Identify Novel Classes
    NOVEL_CLASSES = ["Paribar_Family", "Chikissha_Treatment_Lefthand", "Chikissha_Treatment_Righthand", "Bhalo_Fine"]
    novel_classes_present = [c for c in NOVEL_CLASSES if c in all_classes]

    base_num_classes = len(all_classes) - len(novel_classes_present)
    model = EmbeddingSTGCN(base_num_classes, Graph().A).to(dml)
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=dml, weights_only=False))
    model.eval()

    test_dataset = CleanBanglaDataset(TEST_DIR, class_to_id)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    true_strict = []
    pred_strict = []
    pseudo_probs = []

    print("🧠 Extracting test set distances...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(dml)
            embeddings = model(inputs) 
            distances = torch.cdist(embeddings, prototypes_tensor, p=2)
            
            probs = F.softmax(-distances, dim=1).cpu().numpy()
            min_distances, pred_ids = torch.min(distances, dim=1)
            
            for i in range(labels.size(0)):
                true_strict.append(id_to_class[labels[i].item()])
                pred_strict.append(id_to_class[pred_ids[i].item()])
                pseudo_probs.append(probs[i])

    # Apply Logic Mappings
    true_relaxed = [map_to_relaxed(l) for l in true_strict]
    pred_relaxed = [map_to_relaxed(p) for p in pred_strict]
    true_base = [map_to_base_sign(l) for l in true_strict]
    pred_base = [map_to_base_sign(p) for p in pred_strict]

    # Map Novel Classes for each logic
    novel_strict = novel_classes_present
    novel_relaxed = list(set([map_to_relaxed(c) for c in novel_strict]))
    novel_base = list(set([map_to_base_sign(c) for c in novel_strict]))

    def eval_and_save(y_t, y_p, name_prefix):
        p, r, f1, _ = precision_recall_fscore_support(y_t, y_p, average='macro', zero_division=0)
        labels_unique = sorted(list(set(y_t) | set(y_p)))
        cm = confusion_matrix(y_t, y_p, labels=labels_unique)
        pd.DataFrame(cm, index=labels_unique, columns=labels_unique).to_csv(os.path.join(EVAL_DIR, f"{name_prefix}_Confusion_Matrix.csv"))
        
        with open(os.path.join(EVAL_DIR, f"{name_prefix}_Metrics.txt"), "w", encoding="utf-8") as f:
            f.write(f"=== {name_prefix.upper()} RESULTS ===\n")
            f.write(classification_report(y_t, y_p, zero_division=0))

    print("📊 Generating Standard Matrices...")
    eval_and_save(true_strict, pred_strict, "Strict_Logic")
    eval_and_save(true_relaxed, pred_relaxed, "Relaxed_Logic")
    eval_and_save(true_base, pred_base, "Base_Sign_Logic")

    # --- BASE VS NOVEL COMPARISON REPORT ---
    print("📝 Generating Multi-Logic Base vs Novel Report...")
    
    with open(os.path.join(EVAL_DIR, "Base_vs_Novel_Comparison.txt"), "w", encoding="utf-8") as f:
        f.write("=== FEW-SHOT CLASS-INCREMENTAL LEARNING COMPARISON ===\n\n")
        
        def write_comparison_section(y_t, y_p, novel_list, title):
            y_t_arr = np.array(y_t)
            y_p_arr = np.array(y_p)
            
            base_mask = np.array([t not in novel_list for t in y_t_arr])
            novel_mask = np.array([t in novel_list for t in y_t_arr])
            
            global_acc = np.mean(y_t_arr == y_p_arr) * 100 if len(y_t_arr) > 0 else 0
            base_acc = np.mean(y_t_arr[base_mask] == y_p_arr[base_mask]) * 100 if base_mask.sum() > 0 else 0
            novel_acc = np.mean(y_t_arr[novel_mask] == y_p_arr[novel_mask]) * 100 if novel_mask.sum() > 0 else 0
            
            f.write(f"[{title.upper()}]\n")
            f.write("1. GLOBAL MODEL PERFORMANCE\n")
            f.write(f"Global Top-1 Accuracy: {global_acc:.2f}%\n\n")
            
            f.write("2. BASE CLASSES\n")
            f.write(f"Accuracy: {base_acc:.2f}%\n")
            f.write(f"Total Samples Tested: {base_mask.sum()}\n\n")
            
            f.write("3. NOVEL CLASSES (30-Shot)\n")
            f.write(f"Overall Novel Accuracy: {novel_acc:.2f}%\n")
            f.write(f"Total Novel Samples Tested: {novel_mask.sum()}\n\n")
            f.write("Novel Class Breakdown:\n")
            for n_c in sorted(novel_list):
                cls_mask = y_t_arr == n_c
                if cls_mask.sum() > 0:
                    cls_acc = np.mean(y_t_arr[cls_mask] == y_p_arr[cls_mask]) * 100
                    f.write(f" - {n_c}: {cls_acc:.2f}% (Tested on {cls_mask.sum()} samples)\n")
            f.write("\n" + "="*60 + "\n\n")

        write_comparison_section(true_strict, pred_strict, novel_strict, "STRICT LOGIC")
        write_comparison_section(true_relaxed, pred_relaxed, novel_relaxed, "RELAXED LOGIC")
        write_comparison_section(true_base, pred_base, novel_base, "BASE SIGN LOGIC")

        # --- ROC AUC CALCULATION ---
        try:
            lb = LabelBinarizer()
            y_true_bin = lb.fit_transform(true_strict)
            pseudo_probs = np.array(pseudo_probs)
            macro_roc_auc = roc_auc_score(y_true_bin, pseudo_probs, average='macro', multi_class='ovr')
            
            # Save Sample Graph
            plt.figure(figsize=(10, 8))
            for i, cls in enumerate(lb.classes_[:10]):  
                fpr, tpr, _ = roc_curve(y_true_bin[:, i], pseudo_probs[:, i])
                plt.plot(fpr, tpr, lw=2, label=f'{cls} (area = {auc(fpr, tpr):.2f})')
            plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title('OVR ROC Curve (Strict Logic Sample)')
            plt.legend(loc="lower right")
            plt.savefig(os.path.join(EVAL_DIR, "ROC_AUC_Curve_Sample.png"))
            plt.close()
            
            f.write(f"4. OVERALL ROC AUC SCORE (Strict OVR): {macro_roc_auc:.4f}\n")
        except Exception as e:
            f.write(f"4. ROC AUC SCORE Error: {e}\n")

    print("\n✅ EVALUATION MATRICES SUCCESSFULLY GENERATED!")
    print(f"Check the folder: {EVAL_DIR}")

if __name__ == "__main__":
    generate_matrices()


🔍 Scanning available DirectML GPUs...
   -> GPU 0: AMD Radeon(TM) Graphics 
   -> GPU 1: AMD Radeon RX 7900 GRE 
✅ Selected GPU 1: AMD Radeon RX 7900 GRE 

🚀 STARTING ADVANCED EVALUATION ON NEW DATABASE...
🧠 Extracting test set distances...
📊 Generating Standard Matrices...
📝 Generating Multi-Logic Base vs Novel Report...

✅ EVALUATION MATRICES SUCCESSFULLY GENERATED!
Check the folder: C:\Users\User\Documents\Personal Akams\Thesis\1.Making the model\Baselinemodel Code Experiments\experimental Database Maker\Evaluation Matrices
